In [ ]:
import os

# Move one directory back
os.chdir('..')

# Verify the current working directory
print("Current working directory:", os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, lfilter
import biosppy
from scipy.fftpack import fft

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import biosppy
from biosppy.signals import ppg
from biosppy.signals.tools import filter_signal
import plotly.graph_objs as go
from scipy.signal import find_peaks, butter, filtfilt
import librosa
import neurokit2 as nk

In [ ]:
from ppg_preprocess import generate_synthetic_ppg, add_noise, process_ppg_with_biosppy, process_ppg,is_ppg_signal 
from scipy.signal import butter, filtfilt
from ppg_preprocess import butter_bandpass, bandpass_filter,plot_fft

# LOAD FILE HERE

In [ ]:
df = pd.read_csv('/home/yanwar/Desktop/Sleep/data/adhd_train/adhd-KKI_003-left-sync.csv')
#df = pd.read_csv('/home/yanwar/Desktop/Sleep/data/test/california-00000192-right-sync.csv')

df['sleep_label'] = df['sleep_stage'].apply(lambda x: 0 if x == 'WK' else 1)  # 0: awake, 1: sleep
df = df.dropna(subset=['ledIR', 'ledRed', 'ledGreen', 'sleep_label'])
df['formatted_time'] = pd.to_datetime(df['unixTimes'], unit='ms')

In [ ]:
df

In [ ]:
fs = 25
lowcut = 0.2
highcut = 5.0

df['ledGreen_filtered']=bandpass_filter(df['ledGreen'].dropna(), lowcut, highcut, fs)
df['ledRed_filtered']=bandpass_filter(df['ledRed'].dropna(), lowcut, highcut, fs)
df['ledIR_filtered']=bandpass_filter(df['ledIR'].dropna(), lowcut, highcut, fs)

In [ ]:
ppg_results = biosppy.signals.ppg.ppg(signal=df['ledGreen'].values, sampling_rate=25, show=True)

# Extract the filtered signal and heart rate
heart_rate = ppg_results['heart_rate']
print("Average heart_rate :", np.mean(heart_rate))
print("heart_rate",heart_rate)

In [ ]:
# Create Plotly figure
fig = go.Figure()

# Add trace for processed ledGreen signal
fig.add_trace(go.Scatter(x=df['unixTimes'], y=df['ledGreen_filtered'],
                         mode='lines', name='Processed ledGreen'))

fig.add_trace(go.Scatter(x=df['unixTimes'], y=df['ledGreen'],
                         mode='lines', name='Processed ledGreen'))

fig.add_trace(go.Scatter(
    x=df[df['sleep_label'] == 1]['unixTimes'], 
    y=df[df['sleep_label'] == 1]['ledGreen_filtered'], 
    mode='lines',
    line=dict(color='blue', dash='dot'),  # Blue dashed line for sleep periods
    name='Sleep Periods (Label 1)'
))

# Highlight wake periods (sleep_label == 0) with another color
fig.add_trace(go.Scatter(
    x=df[df['sleep_label'] == 0]['unixTimes'], 
    y=df[df['sleep_label'] == 0]['ledGreen_filtered'], 
    mode='lines',
    line=dict(color='orange', dash='dash'),  # Orange dashed line for wake periods
    name='Wake Periods (Label 0)'
))
# Update layout
fig.update_layout(title='Processed ledGreen Signal with Sleep Periods',
                  xaxis_title='Time (unixTimes)',
                  yaxis_title='Signal Value',
                  hovermode='x unified')

# Show the plot
fig.show()


In [ ]:
plot_fft(df['ledGreen_filtered'], fs=25, window='hann', n_fft=512, title='FFT of original PPG Signal')

# SIGNAL QUALITY CLASSIFICATION

In [ ]:
from ppg_preprocess import *

In [ ]:
SAMPLING_RATE = 25
LOWCUT = 0.2
HIGHCUT = 5.0
SYNT_HEART_RATE = 70
SYNT_DURATION = 30
N_FFT=512
NOISE_LEVEL = 0.3

# CLASSIFICATION WINDOW AND STEP
SEGMENT_LENGTH_S = 20  # 20 seconds per segment
STEP_SIZE_S = 5        # 5 seconds step size

In [ ]:
# SYNTHETIC PPG
synthetic_ppg = generate_synthetic_ppg(SYNT_DURATION, SAMPLING_RATE, SYNT_HEART_RATE)
synthetic_ppg=bandpass_filter(synthetic_ppg, LOWCUT, HIGHCUT, SAMPLING_RATE)


In [ ]:
#SNR THRESHOLD
snr_threshold = determine_snr_threshold(synthetic_ppg, SAMPLING_RATE, NOISE_LEVEL,'hann',N_FFT)
print("snr_threshold: ",snr_threshold)

In [ ]:
# CLASSIFICATION SEGMENT
segment_length = SEGMENT_LENGTH_S * SAMPLING_RATE  # Segment length in samples
step_size = STEP_SIZE_S * SAMPLING_RATE       
segments = overlapping_windows(df['ledGreen_filtered'].values, segment_length, step_size)

good_bad_segments_overlap = []
frequencies = np.fft.fftfreq(N_FFT, 1/SAMPLING_RATE)[:N_FFT // 2]
signal_band = (frequencies >= LOWCUT) & (frequencies <= HIGHCUT)  # PPG signal band
noise_band = (frequencies < LOWCUT) | (frequencies > HIGHCUT)     # Noise band
for i, segment in enumerate(segments):
    if len(segment) < segment_length:
        continue  # Skip segments that are too short
    #CHECK FOR SNR
    #snr = calculate_snr(segment, SAMPLING_RATE, signal_band, noise_band, 'hann', N_FFT)
    snr = calculate_snr_around_peaks(segment, SAMPLING_RATE, signal_band, noise_band, 'hann', N_FFT)
    
    #CHECK FOR PEAKS
    is_ppg, details = is_ppg_signal(segment, sampling_rate=SAMPLING_RATE)
    
    if 'hrv_metrics' in details:
        #if details['hrv_metrics']['SDNN']<50 or  details['hrv_metrics']['SDNN']>600:
        #    is_ppg=False
        #else:
        #    is_ppg=True
        if details['hrv_metrics']['RMSSD']<20 or  details['hrv_metrics']['SDNN']>190:
            is_ppg=False
        else:
            is_ppg=True
        if details['hrv_metrics']['mean_hr']<55 or details['hrv_metrics']['mean_hr']>160:
            is_ppg=False
        else:
            is_ppg=True
        
    #if is_ppg:
    #    if 1000*(details['hrv_metrics']['RMSSD'])>300 or 1000*(details['hrv_metrics']['RMSSD'])<10:
    #        is_ppg=False
        #if details['hrv_metrics']['SDNN']<0.05 or  details['hrv_metrics']['SDNN']>0.6:
        #    is_ppg=False
            
        #IF RMMSD IF ITS TOO HIGH LIKE MORE THAN 150-200
        # HR ANYTHING AROUND 150 IS BAD
    
    quality = 'good' if snr >= snr_threshold and is_ppg  else 'bad'
    start_idx = i * step_size
    end_idx = start_idx + segment_length
    good_bad_segments_overlap.append((quality, start_idx, end_idx))
    


In [ ]:
# Aggregate the classifications for each signal value
aggregated_score = aggregate_classifications(good_bad_segments_overlap, 
                                             len(df['ledGreen_filtered']), 
                                             segment_length, step_size)

In [ ]:
good_mask = aggregated_score >= 0
df_filtered = df[good_mask].reset_index(drop=True)

print("Original DataFrame shape:", df.shape)
print("Filtered DataFrame shape:", df_filtered.shape)

In [ ]:
good_segments = df[good_mask]
bad_segments = df[~good_mask]

In [ ]:
# Ensure unixTimes is numeric
df['unixTimes'] = pd.to_numeric(df['unixTimes'], errors='coerce')

# Plot using Plotly
fig = go.Figure()

# Add the original signal trace
fig.add_trace(go.Scatter(x=df['unixTimes'], y=df['ledGreen_filtered'], mode='lines', name='Actual PPG Signal'))

# Add the good segments
fig.add_trace(go.Scatter(x=good_segments['unixTimes'], y=good_segments['ledGreen_filtered'],
                         mode='lines', line=dict(color='green'), name='Good Segments'))

# Add the bad segments
fig.add_trace(go.Scatter(x=bad_segments['unixTimes'], y=bad_segments['ledGreen_filtered'],
                         mode='lines', line=dict(color='red'), name='Bad Segments'))

# Highlight sleep periods (sleep_label == 1) with a specific color
fig.add_trace(go.Scatter(
    x=df[df['sleep_label'] == 1]['unixTimes'], 
    y=df[df['sleep_label'] == 1]['ledGreen_filtered'], 
    mode='lines',
    line=dict(color='blue', dash='dot'),  # Blue dashed line for sleep periods
    name='Sleep Periods (Label 1)'
))

# Highlight wake periods (sleep_label == 0) with another color
fig.add_trace(go.Scatter(
    x=df[df['sleep_label'] == 0]['unixTimes'], 
    y=df[df['sleep_label'] == 0]['ledGreen_filtered'], 
    mode='lines',
    line=dict(color='orange', dash='dash'),  # Orange dashed line for wake periods
    name='Wake Periods (Label 0)'
))

# Update layout
fig.update_layout(
    title='PPG Signal with Good and Bad Segments and Sleep Labels',
    xaxis=dict(
        title='Time (Unix Timestamps)',
        tickformat='d'  # Ensures Unix timestamps are displayed in full
    ),
    yaxis_title='PPG Signal Amplitude',
    hovermode='x unified'
)

# Show the plot
fig.show()

In [ ]:
# PLOT SMALL SEGMENT

In [ ]:
start_unix_time = 1648112450000  # Start time in milliseconds
end_unix_time = 1648112470000    # End time in milliseconds

# Ensure that both start and end times are in milliseconds
# If they are in seconds, use the values directly
# Calculate the window length in milliseconds
window_length_ms = end_unix_time - start_unix_time

# Convert the window length to seconds
window_length_s = window_length_ms / 1000.0  # Converts milliseconds to seconds

print(f"Window length in milliseconds: {window_length_ms} ms")
print(f"Window length in seconds: {window_length_s} s")



time_filtered_df = df[(df['unixTimes'] >= start_unix_time) & (df['unixTimes'] <= end_unix_time)]
time_filtered_ppg_signal = time_filtered_df['ledGreen_filtered'].values


is_ppg, details = is_ppg_signal(time_filtered_ppg_signal, sampling_rate=25)
print(details)

peaks, info = nk.ppg_peaks(time_filtered_ppg_signal, sampling_rate=25, method="elgendi", show=True)

#print(time_filtered_ppg_signal)
#ppg_signal = df['ledGreen_filtered'].values
plot_fft(time_filtered_ppg_signal, fs=25, window='hann', n_fft=512, title='FFT of actual PPG Signal')


fig = go.Figure()
fig.add_trace(go.Scatter(x=time_filtered_df['unixTimes'], y=time_filtered_ppg_signal, mode='lines', name='Filtered PPG Signal'))
fig.update_layout(
    title='Filtered PPG Signal',
    xaxis_title='Time (Unix Timestamps)',
    yaxis_title='PPG Signal Amplitude',
    hovermode='x unified'
)
fig.show()


In [ ]:
# Ensure unixTimes is numeric
df['unixTimes'] = pd.to_numeric(df['unixTimes'], errors='coerce')

# Initialize figure
fig = go.Figure()

# Add the original signal trace
fig.add_trace(go.Scatter(x=df['unixTimes'], y=df['ledGreen_filtered'], mode='lines', name='Actual PPG Signal'))

# Add the good segments
fig.add_trace(go.Scatter(x=good_segments['unixTimes'], y=good_segments['ledGreen_filtered'],
                         mode='lines', line=dict(color='green'), name='Good Segments'))

# Add the bad segments
fig.add_trace(go.Scatter(x=bad_segments['unixTimes'], y=bad_segments['ledGreen_filtered'],
                         mode='lines', line=dict(color='red'), name='Bad Segments'))

# Function to plot segments with a continuous line but different colors based on label
def plot_sleep_segments(df, fig):
    # Initialize the first index
    start_idx = 0
    
    # Iterate through the dataframe to split into different segments
    for i in range(1, len(df)):
        # Check if the label changes between points
        if df['sleep_label'].iloc[i] != df['sleep_label'].iloc[i-1]:
            # Plot the segment before the label changes
            fig.add_trace(go.Scatter(
                x=df['unixTimes'].iloc[start_idx:i],
                y=df['ledGreen_filtered'].iloc[start_idx:i],
                mode='lines',
                line=dict(color='blue' if df['sleep_label'].iloc[i-1] == 1 else 'orange'),
                name='Sleep/Wake Periods'
            ))
            # Update the start index
            start_idx = i
    
    # Plot the last segment
    fig.add_trace(go.Scatter(
        x=df['unixTimes'].iloc[start_idx:],
        y=df['ledGreen_filtered'].iloc[start_idx:],
        mode='lines',
        line=dict(color='blue' if df['sleep_label'].iloc[start_idx] == 1 else 'orange'),
        name='Sleep/Wake Periods'
    ))

# Call the function to add sleep/wake segments
plot_sleep_segments(df, fig)

# Update layout
fig.update_layout(
    title='PPG Signal with Good and Bad Segments and Sleep Labels',
    xaxis=dict(
        title='Time (Unix Timestamps)',
        tickformat='d'  # Ensures Unix timestamps are displayed in full
    ),
    yaxis_title='PPG Signal Amplitude',
    hovermode='x unified'
)

# Show the plot
fig.show()

# PLOT WITH SPECTRUM

In [ ]:
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable, coolwarm

# Normalize the aggregated scores for color mapping
score_min, score_max = aggregated_score.min(), aggregated_score.max()
norm = Normalize(vmin=score_min, vmax=score_max)
mappable = ScalarMappable(norm=norm, cmap=coolwarm.reversed())
colors = mappable.to_rgba(aggregated_score)

In [ ]:

# Plot using Plotly
fig = go.Figure()

# Add the original signal trace (with lower opacity for background)
fig.add_trace(go.Scatter(x=df['unixTimes'], y=df['ledGreen_filtered'], mode='lines', name='Actual PPG Signal', line=dict(color='lightgray', width=1)))

# Group and add colored segments based on the aggregated scores
previous_color = colors[0]
x = [df['unixTimes'].iloc[0]]
y = [df['ledGreen_filtered'].iloc[0]]

for i in range(1, len(df)):
    current_color = colors[i]
    if not np.array_equal(current_color, previous_color):
        # Add trace for the previous segment
        color_rgba = f'rgba({previous_color[0]*255}, {previous_color[1]*255}, {previous_color[2]*255}, {previous_color[3]})'
        fig.add_trace(go.Scatter(x=x, y=y, mode='lines', line=dict(color=color_rgba, width=2), showlegend=False))
        # Start a new segment
        x = []
        y = []
        previous_color = current_color
    x.append(df['unixTimes'].iloc[i])
    y.append(df['ledGreen_filtered'].iloc[i])

# Add the last segment
color_rgba = f'rgba({previous_color[0]*255}, {previous_color[1]*255}, {previous_color[2]*255}, {previous_color[3]})'
fig.add_trace(go.Scatter(x=x, y=y, mode='lines', line=dict(color=color_rgba, width=2), showlegend=False))

# Update layout
fig.update_layout(
    title='PPG Signal with Aggregated Classification Color Mapping (Grouped)',
    xaxis=dict(
        title='Time (Unix Timestamps)',
        tickformat='d'  # Ensures the Unix timestamps are displayed in full
    ),
    yaxis_title='PPG Signal Amplitude',
    hovermode='x unified'
)

# Show the plot
fig.show()

# PLOT SMALL SEGMENT

In [ ]:
start_unix_time = 1698720745000  # Start time in milliseconds
end_unix_time = 1698720760000    # End time in milliseconds

# Ensure that both start and end times are in milliseconds
# If they are in seconds, use the values directly
# Calculate the window length in milliseconds
window_length_ms = end_unix_time - start_unix_time

# Convert the window length to seconds
window_length_s = window_length_ms / 1000.0  # Converts milliseconds to seconds

print(f"Window length in milliseconds: {window_length_ms} ms")
print(f"Window length in seconds: {window_length_s} s")



time_filtered_df = df[(df['unixTimes'] >= start_unix_time) & (df['unixTimes'] <= end_unix_time)]
time_filtered_ppg_signal = time_filtered_df['ledGreen_filtered'].values

peaks, info = nk.ppg_peaks(time_filtered_ppg_signal, sampling_rate=25, method="elgendi", show=True)

#print(time_filtered_ppg_signal)
#ppg_signal = df['ledGreen_filtered'].values
plot_fft(time_filtered_ppg_signal, fs=sampling_rate, window='hann', n_fft=512, title='FFT of actual PPG Signal')


fig = go.Figure()
fig.add_trace(go.Scatter(x=time_filtered_df['unixTimes'], y=time_filtered_ppg_signal, mode='lines', name='Filtered PPG Signal'))
fig.update_layout(
    title='Filtered PPG Signal',
    xaxis_title='Time (Unix Timestamps)',
    yaxis_title='PPG Signal Amplitude',
    hovermode='x unified'
)
fig.show()
